In [ ]:
####### Autoencoder

from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
import numpy as np

# Génération de données factices (normal + anomalies)
np.random.seed(42)
normal_data = np.random.normal(0, 1, (1000, 10))  # Données normales
anomaly_data = np.random.uniform(-5, 5, (20, 10))  # Données d'anomalies

# Construction de l'autoencodeur
input_dim = normal_data.shape[1]
encoding_dim = 5  # Dimension de la représentation latente

input_layer = Input(shape=(input_dim,))
encoded = Dense(encoding_dim, activation='relu')(input_layer)
decoded = Dense(input_dim, activation='sigmoid')(encoded)

autoencoder = Model(input_layer, decoded)
autoencoder.compile(optimizer='adam', loss='mse')

# Entraînement de l'autoencodeur sur les données normales uniquement
autoencoder.fit(normal_data, normal_data, epochs=50, batch_size=32, shuffle=True, validation_split=0.1)

# Reconstruction des données
reconstructed_normal_data = autoencoder.predict(normal_data)
reconstructed_anomaly_data = autoencoder.predict(anomaly_data)

# Calcul de l'erreur de reconstruction (par exemple, erreur quadratique moyenne)
mse_normal = np.mean(np.square(normal_data - reconstructed_normal_data), axis=1)
mse_anomaly = np.mean(np.square(anomaly_data - reconstructed_anomaly_data), axis=1)

# Identification des anomalies en utilisant un seuil d'erreur
threshold = np.mean(mse_normal) + 3 * np.std(mse_normal)
detected_anomalies = anomaly_data[mse_anomaly > threshold]

print("Anomalies détectées :")
print(detected_anomalies)


In [ ]:
####### Autoencoder variationel

import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.losses import MeanSquaredError, KLDivergence

# Génération de données factices pour l'exemple
np.random.seed(42)
normal_data = np.random.normal(0, 1, (1000, 10))  # Données normales
anomaly_data = np.random.uniform(-5, 5, (20, 10))  # Données d'anomalies

# Fonction pour créer le VAE
def create_vae(input_dim, latent_dim):
    # Encodeur
    input_layer = Input(shape=(input_dim,))
    hidden = Dense(15, activation='relu')(input_layer)
    z_mean = Dense(latent_dim)(hidden)
    z_log_var = Dense(latent_dim)(hidden)

    # Fonction d'échantillonnage dans l'espace latent
    def sampling(args):
        z_mean, z_log_var = args
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

    # Utilisation de Lambda pour l'échantillonnage
    z = Lambda(sampling)([z_mean, z_log_var])

    # Décodeur
    decoder_hidden = Dense(15, activation='relu')(z)
    output_layer = Dense(input_dim)(decoder_hidden)

    # Modèle VAE
    vae = Model(input_layer, output_layer)

    # Calcul de la perte VAE
    reconstruction_loss = MeanSquaredError()(input_layer, output_layer)
    kl_loss = -0.5 * tf.reduce_mean(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
    vae_loss = reconstruction_loss + kl_loss

    vae.add_loss(vae_loss)
    vae.compile(optimizer='adam')

    return vae

# Création et entraînement du VAE sur les données normales uniquement
input_dim = normal_data.shape[1]
latent_dim = 5  # Dimension de l'espace latent
vae_model = create_vae(input_dim, latent_dim)
vae_model.fit(normal_data, epochs=50, batch_size=32, shuffle=True, validation_split=0.1)

# Reconstruction de toutes les données (normales + anomalies)
reconstructed_normal_data = vae_model.predict(normal_data)
reconstructed_anomaly_data = vae_model.predict(anomaly_data)

# Calcul de l'erreur de reconstruction (par exemple, erreur quadratique moyenne)
mse_normal = np.mean(np.square(normal_data - reconstructed_normal_data), axis=1)
mse_anomaly = np.mean(np.square(anomaly_data - reconstructed_anomaly_data), axis=1)

# Détection d'anomalies en utilisant un seuil d'erreur
threshold = np.mean(mse_normal) + 3 * np.std(mse_normal)
detected_anomalies = anomaly_data[mse_anomaly > threshold]

print("Anomalies détectées :")
print(detected_anomalies)


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.losses import MeanSquaredError, KLDivergence

# Génération de données factices pour l'exemple
np.random.seed(42)
normal_data = np.random.normal(0, 1, (1000, 10))  # Données normales

# Fonction pour créer le VAE
def create_vae(input_dim, latent_dim):
    # Encodeur
    input_layer = Input(shape=(input_dim,))
    hidden = Dense(15, activation='relu')(input_layer)
    z_mean = Dense(latent_dim)(hidden)
    z_log_var = Dense(latent_dim)(hidden)

    # Fonction d'échantillonnage dans l'espace latent
    def sampling(args):
        z_mean, z_log_var = args
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

    # Utilisation de Lambda pour l'échantillonnage
    z = Lambda(sampling)([z_mean, z_log_var])

    # Décodeur
    decoder_hidden = Dense(15, activation='relu')(z)
    output_layer = Dense(input_dim)(decoder_hidden)

    # Modèle VAE
    vae = Model(input_layer, output_layer)

    # Calcul de la perte VAE
    reconstruction_loss = MeanSquaredError()(input_layer, output_layer)
    kl_loss = -0.5 * tf.reduce_mean(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
    vae_loss = reconstruction_loss + kl_loss

    vae.add_loss(vae_loss)
    vae.compile(optimizer='adam')

    return vae

# Création et entraînement du VAE sur les données normales uniquement
input_dim = normal_data.shape[1]
latent_dim = 5  # Dimension de l'espace latent
vae_model = create_vae(input_dim, latent_dim)
vae_model.fit(normal_data, epochs=50, batch_size=32, shuffle=True, validation_split=0.1)

# Reconstruction de toutes les données (normales uniquement)
reconstructed_data = vae_model.predict(normal_data)

# Calcul de l'erreur de reconstruction (par exemple, erreur quadratique moyenne)
mse = np.mean(np.square(normal_data - reconstructed_data), axis=1)

# Détection d'anomalies en utilisant un seuil d'erreur
threshold = np.mean(mse) + 3 * np.std(mse)
detected_anomalies = normal_data[mse > threshold]

print("Anomalies détectées :")
print(detected_anomalies)
